# 第10章：全流程实战

## 本章目标
- 将 Ch1-9 的知识串联成完整的端到端 pipeline
- 从数据准备到模型对齐到量化部署的完整流程
- 最佳实践 checklist

## 全流程概览
```
数据准备 → Tokenizer → Pretrain → SFT → DPO → 量化 → 部署
```
注：本 chapter 在 Colab T4 上完整运行，使用 Qwen2.5-0.5B 小模型演示。

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch transformers trl peft datasets accelerate bitsandbytes
else:
    print("本地环境运行，请确保已按 intro.md 配置好环境")

In [ ]:
from datasets import load_dataset

# Step 1: 数据准备
raw_data = load_dataset("tatsu-lab/alpaca", split="train[:2000]")
preference_data = load_dataset("Anthropic/hh-rlhf", split="train[:1000]")

print(f"SFT 数据: {len(raw_data)} 样本")
print(f"偏好数据: {len(preference_data)} 样本")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
import torch

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Step 2: 加载 Base Model
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)
print(f"Base model loaded: {model_name}")
print(f"参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

In [ ]:
from trl import SFTTrainer, SFTConfig

# Step 3: SFT
def format_sft(example):
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"}

sft_data = raw_data.map(format_sft)

# 应用 LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

sft_trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir="./pipeline_sft", num_train_epochs=1,
        per_device_train_batch_size=4, learning_rate=2e-4,
        gradient_accumulation_steps=4,
        max_seq_length=512, logging_steps=50, save_strategy="no", report_to="none",
    ),
    train_dataset=sft_data,
    processing_class=tokenizer,
)
sft_trainer.train()
print("Step 3: SFT 完成 ✓")

In [ ]:
# 预处理 hh-rlhf 数据：将原始对话文本转为 message 列表格式
def extract_dialogue(text):
    """将 hh-rlhf 的原始对话文本拆分为 message 列表"""
    messages = []
    parts = text.split("\n\nHuman: ")
    for part in parts[1:]:
        messages.append({"role": "user", "content": part.split("\n\nAssistant: ")[0].strip()})
        assistant_parts = part.split("\n\nAssistant: ")
        if len(assistant_parts) > 1:
            messages.append({"role": "assistant", "content": assistant_parts[1].strip()})
    return messages

preference_data = preference_data.map(lambda x: {
    "chosen": extract_dialogue(x["chosen"]),
    "rejected": extract_dialogue(x["rejected"]),
})
print("偏好数据预处理完成 ✓")

In [ ]:
from trl import DPOTrainer, DPOConfig

# Step 4: DPO
ref_model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=DPOConfig(
        output_dir="./pipeline_dpo", num_train_epochs=1,
        per_device_train_batch_size=2, learning_rate=5e-5,
        max_length=512, beta=0.1, logging_steps=50, save_strategy="no", report_to="none",
    ),
    train_dataset=preference_data,
    processing_class=tokenizer,
)
dpo_trainer.train()
print("Step 4: DPO 完成 ✓")

In [ ]:
# Step 5: 保存 + 量化
final_model_path = "./pipeline_final"
# 先合并 LoRA adapter 到 base model，否则只保存 adapter 权重
merged_model = model.merge_and_unload()
merged_model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"模型已保存到 {final_model_path}")

if torch.cuda.is_available():
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    quantized = AutoModelForCausalLM.from_pretrained(
        final_model_path, quantization_config=bnb_config, device_map="auto"
    )
    print("4-bit 量化完成 ✓")
else:
    print("量化需要 GPU，跳过。")

In [ ]:
# Step 6: 评估
test_prompts = [
    "What is machine learning?",
    "Explain the concept of attention in transformers.",
    "Write a short poem about AI.",
]

for prompt in test_prompts:
    inputs = tokenizer(f"### Instruction:\n{prompt}\n\n### Response:\n", return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {prompt}")
    print(f"A: {response}\n")

## Best Practices Checklist

### 数据
- 数据质量 > 数据数量
- 去重、清洗、去毒
- SFT 数据覆盖目标任务的 diversity

### 训练
- 先 SFT 再 DPO/RLHF
- LoRA rank 根据任务复杂度调整 (8-64)
- Learning rate: SFT ~2e-4, DPO ~5e-5
- 始终监控 train/val loss 曲线

### 评估
- 自动评估（perplexity, win rate）
- 人工评估（golden set）
- 安全性测试（red teaming）

### 部署
- 先量化再部署（4-bit 或 8-bit）
- 使用 KV cache 加速推理
- 考虑 vLLM / TensorRT-LLM 等推理引擎

## 延伸阅读

恭喜完成全部 10 章的学习！你已经从零掌握了 LLM 训练的完整链路。

### 推荐继续学习的方向
- [nanoGPT](https://github.com/karpathy/nanoGPT) — 从零实现 GPT 训练
- [litgpt](https://github.com/Lightning-AI/litgpt) — 生产级 LLM 训练框架
- [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory) — 统一的 LLM 微调平台
- [Axolotl](https://github.com/OpenAccess-AI-Collective/axolotl) — 高质量 LLM 微调工具
- [vLLM](https://github.com/vllm-project/vllm) — 高性能推理引擎